# Joshi Part 7: Convergence & Variance Reduction (Rust)

Rust kernel version of `09_joshi_convergence.ipynb`.

## Setup

In [ ]:
:dep RustQuant = { path = "../crates/RustQuant" }
:dep time = { version = "0.3", features = ["macros"] }
:dep rand = "0.8"

In [ ]:
use std::f64::consts::PI;
use time::macros::date;
use RustQuant::instruments::options::*;

let spot = 100.0_f64;
let strike = 100.0;
let rate = 0.05;
let vol = 0.20;
let t = 1.0;
let df = (-rate * t).exp();

let bsm = BlackScholesMertonBuilder::default()
    .underlying_price(spot).strike_price(strike).volatility(vol)
    .risk_free_rate(rate).cost_of_carry(rate)
    .expiration_date(date!(2027 - 03 - 22)).option_type(TypeFlag::Call)
    .build().unwrap();
let exact = bsm.price();
println!("Exact BS Price: {:.6}", exact);

## 1. Convergence Table (Standard MC)

In [ ]:
struct Stats {
    sum: f64,
    sum_sq: f64,
    count: usize,
}

impl Stats {
    fn new() -> Self { Self { sum: 0.0, sum_sq: 0.0, count: 0 } }
    fn add(&mut self, v: f64) { self.sum += v; self.sum_sq += v*v; self.count += 1; }
    fn mean(&self) -> f64 { self.sum / self.count as f64 }
    fn stderr(&self) -> f64 {
        let n = self.count as f64;
        ((self.sum_sq / n - (self.sum / n).powi(2)) / n).sqrt()
    }
}

let mut rng = rand::thread_rng();
let mut stats = Stats::new();
let total = 1_048_576_usize;
let mut next = 256_usize;

println!("{:>10} {:>10} {:>10} {:>10} {:>10}", "Paths", "Price", "Std Err", "95% CI±", "Error");
println!("{}", "-".repeat(50));

for _ in 0..total {
    let u1: f64 = rand::Rng::gen(&mut rng);
    let u2: f64 = rand::Rng::gen(&mut rng);
    let z = (-2.0*u1.ln()).sqrt() * (2.0*PI*u2).cos();
    let s_t = spot * ((rate - 0.5*vol*vol)*t + vol*t.sqrt()*z).exp();
    stats.add(df * (s_t - strike).max(0.0));

    if stats.count == next {
        let se = stats.stderr();
        println!("{:>10} {:>10.4} {:>10.6} {:>10.6} {:>+10.4}",
            stats.count, stats.mean(), se, 1.96*se, stats.mean()-exact);
        next *= 2;
    }
}
let se = stats.stderr();
println!("{:>10} {:>10.4} {:>10.6} {:>10.6} {:>+10.4}",
    stats.count, stats.mean(), se, 1.96*se, stats.mean()-exact);

## 2. Antithetic Variates

For each $Z$, also evaluate with $-Z$. Reduces variance for convex payoffs.

In [ ]:
let mut stats_av = Stats::new();
let mut next = 256_usize;

println!("{:>10} {:>10} {:>10} {:>10} {:>10}", "Paths", "Price", "Std Err", "95% CI±", "Error");
println!("{}", "-".repeat(50));

for _ in 0..total {
    let u1: f64 = rand::Rng::gen(&mut rng);
    let u2: f64 = rand::Rng::gen(&mut rng);
    let z = (-2.0*u1.ln()).sqrt() * (2.0*PI*u2).cos();

    let s_pos = spot * ((rate - 0.5*vol*vol)*t + vol*t.sqrt()*z).exp();
    let s_neg = spot * ((rate - 0.5*vol*vol)*t + vol*t.sqrt()*(-z)).exp();
    let payoff = 0.5 * ((s_pos - strike).max(0.0) + (s_neg - strike).max(0.0));
    stats_av.add(df * payoff);

    if stats_av.count == next {
        let se = stats_av.stderr();
        println!("{:>10} {:>10.4} {:>10.6} {:>10.6} {:>+10.4}",
            stats_av.count, stats_av.mean(), se, 1.96*se, stats_av.mean()-exact);
        next *= 2;
    }
}
let se = stats_av.stderr();
println!("{:>10} {:>10.4} {:>10.6} {:>10.6} {:>+10.4}",
    stats_av.count, stats_av.mean(), se, 1.96*se, stats_av.mean()-exact);

## 3. RustQuant MC at Various Path Counts

In [ ]:
use RustQuant::instruments::*;
use RustQuant::stochastics::*;

let gbm = GeometricBrownianMotion::new(rate, vol);
let expiry = date!(2027 - 03 - 22);
let vanilla = EuropeanVanillaOption::new(strike, expiry, TypeFlag::Call);

println!("{:>10} {:>10} {:>10}", "Paths", "MC Price", "Error");
println!("{}", "-".repeat(30));
for &n in &[1_000_usize, 5_000, 10_000, 50_000, 100_000, 500_000] {
    let cfg = StochasticProcessConfig::new(
        spot, 0.0, t, 1, StochasticScheme::EulerMaruyama, n, true, None,
    );
    let price = vanilla.price_monte_carlo(&gbm, &cfg, rate);
    println!("{:>10} {:>10.4} {:>+10.4}", n, price, price - exact);
}

println!("\nMC error is O(1/sqrt(N)). Antithetic variates roughly halve the variance.");